# Veriti RAG quick notebook

This notebook shows the project in a very small, step-by-step way: configure the app, ingest the knowledge base, and run a search to inspect the results.

This uses the offline `hash` embedding mode, so it works without a live OpenRouter API key.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
print(f"Working directory: {root}")
assert (root / 'veriti_rag').is_dir(), 'Please open this notebook from the project root folder.'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
root

Working directory: c:\Projects\New folder\veriti_rag_app\veriti_rag_app


WindowsPath('c:/Projects/New folder/veriti_rag_app/veriti_rag_app')

## Step 1: Build the settings and vector store

The project is configured with a local `knowledge_base` folder and a local Chroma store. Using `embeddings='hash'` makes the notebook safe for fast local testing.

In [2]:
from veriti_rag.config import Settings
from veriti_rag.models import HashEmbeddings
from veriti_rag.store import get_vectorstore

s = Settings(
    kb_dir=str(root / 'knowledge_base'),
    index_dir=str(root / 'vector_index'),
    embeddings='hash',
    client_id='maven_demo',
)
vs = get_vectorstore(s, HashEmbeddings())
print(s)
print(vs)

Settings(kb_dir='c:\\Projects\\New folder\\veriti_rag_app\\veriti_rag_app\\knowledge_base', data_dir='data/raw', index_dir='c:\\Projects\\New folder\\veriti_rag_app\\veriti_rag_app\\vector_index', client_id='maven_demo', base_url='https://openrouter.ai/api/v1', llm_model='anthropic/claude-opus-5.5', embed_model='openai/text-embedding-3-small', embeddings='hash', chunk_size=800, chunk_overlap=100, top_k=5, max_sql_attempts=3, max_answer_attempts=2)


## Step 2: Ingest the knowledge base

This runs the ingestion graph: load docs, split them into chunks, and store the chunks in the vector database.

In [3]:
from veriti_rag.ingest import ingest_all

reports = ingest_all(s, vs)
for item in reports:
    print(item)
    print('-' * 40)

{'client': 'maven_demo', 'chunks': 33, 'added': 33, 'deleted': 0, 'unchanged': 0}
----------------------------------------


## Step 3: Run a search and inspect the matches

This is the retrieval step. We search for a business rule phrase and inspect the returned chunk metadata and content.

In [4]:
from veriti_rag.store import search

query = 'won deals missing close value'
results = search(vs, query, 'maven_demo', k=3)
print(f"Query: {query}")
print(f"Number of results: {len(results)}")

for doc, score in results:
    print(f"Score: {score:.3f}")
    print('Metadata:', doc.metadata)
    print('Content preview:')
    print(doc.page_content[:400].replace('\n', ' '))
    print('=' * 80)

Query: won deals missing close value
Number of results: 3
Score: 0.479
Metadata: {'client_id': 'maven_demo', 'version': 'v1', 'doc_type': 'drc_rule', 'chunk_id': 'fd98bee2eb517016', 'source': 'drc_rules.yml', 'rule_id': 'DRC-COMP-002'}
Content preview:
id: DRC-COMP-002 description: Won deals must have a close value table: sales_pipeline column: close_value dimension: completeness severity: critical bsd: revenue_reporting fail_condition: deal_stage = 'Won' AND close_value IS NULL
Score: 0.379
Metadata: {'chunk_id': 'be7f445781873dff', 'rule_id': 'DRC-COMP-002', 'client_id': 'maven_demo', 'doc_type': 'remediation', 'h1': 'Remediation guide', 'h2': 'DRC-COMP-002 Missing close value on won deals', 'version': 'v1', 'source': 'remediation_guide.md'}
Content preview:
# Remediation guide   ## DRC-COMP-002 Missing close value on won deals Back-fill from the billing or contract system. If unavailable, flag the deal for the sales owner to confirm and exclude it from revenue reports until resolved

## Step 4: Try another query or inspect the index

You can change the query below to see how the retrieval behaves for different rule names or phrases.

In [ ]:
for q in ['close value', 'sales pipeline', 'DRC-COMP-002', 'won deals']:
    matches = search(vs, q, 'maven_demo', k=2)
    print(f"\nQuery: {q}\n---")
    for doc, score in matches:
        print(f"{score:.3f} | {doc.metadata.get('source', 'unknown')} | {doc.metadata.get('rule_id', 'n/a')}")
